<a href="https://colab.research.google.com/github/irullah/lda-topic-modeling/blob/main/topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TOPIC MODELLING

In [ ]:
# Install library yang dibutuhkan
# !pip install sastrawi swifter gensim google-api-python-client

import pandas as pd
import numpy as np
import re
import nltk
import swifter
from string import punctuation
from googleapiclient.discovery import build
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Download dependensi NLTK yang WAJIB untuk tokenisasi dan stopwords (FIX ERROR BLOK 12)
nltk.download('punkt')
nltk.download('punkt_tab') # Ini yang menyelesaikan error Anda
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
def get_video_comments(video_id, api_key):
    """
    Fungsi untuk mengambil komentar dan balasan dari video YouTube.
    """
    replies = []
    youtube = build('youtube', 'v3', developerKey=api_key)

    try:
        video_response = youtube.commentThreads().list(part='snippet,replies', videoId=video_id).execute()

        while video_response:
            for item in video_response['items']:
                # Ekstrak komentar utama
                top_comment = item['snippet']['topLevelComment']['snippet']
                replies.append([
                    top_comment['publishedAt'],
                    top_comment['authorDisplayName'],
                    top_comment['textDisplay'],
                    top_comment['likeCount']
                ])

                # Ekstrak balasan (replies) jika ada
                if item['snippet']['totalReplyCount'] > 0 and 'replies' in item:
                    for reply in item['replies']['comments']:
                        reply_snippet = reply['snippet']
                        replies.append([
                            reply_snippet['publishedAt'],
                            reply_snippet['authorDisplayName'],
                            reply_snippet['textDisplay'],
                            reply_snippet['likeCount']
                        ])

            # Cek halaman selanjutnya
            if 'nextPageToken' in video_response:
                video_response = youtube.commentThreads().list(
                    part='snippet,replies',
                    pageToken=video_response['nextPageToken'],
                    videoId=video_id
                ).execute()
            else:
                break

    except Exception as e:
        print(f"Terjadi kesalahan saat mengambil data: {e}")

    return replies

# Eksekusi Pengambilan Data
API_KEY = 'MASUKKAN_API_KEY_ANDA_DI_SINI' # Ganti dengan API Key Anda
VIDEO_ID = "KtntKGlmuZw"

comments_data = get_video_comments(VIDEO_ID, API_KEY)

# Jadikan DataFrame dan simpan
df = pd.DataFrame(comments_data, columns=['Tanggal', 'Nama', 'Komen', 'Like'])
df.to_csv('youtube-comments.csv', index=False)
print(f"Berhasil mengekstrak {len(df)} komentar.")
df.head()

Berhasil mengekstrak 3527 komentar.


,Tanggal,Nama,Komen,Like
0,2026-05-25T07:39:34Z,@Alimukhsinarblit-q9w,Saya pastikan kepala kebo akan nyungsep lagi.,0
1,2026-05-22T03:51:51Z,@poncokendil1733,Singkirkan capres ABG WNI bangsa lain.....pili...,0
2,2025-01-01T02:11:27Z,@NikmatulChasanah-b7j,PK GNJR PK MAGVOD ... Mntab .. 💪💪💪🔥🔥❤️❤️🤟🤟🤟🤟🤟...,0
3,2024-12-27T06:04:26Z,@TarnoLanda-xp4bo,29 tetap pilih figurnya. Yang hobbi korupsi pa...,0
4,2024-12-19T12:46:43Z,@munadimpw8416,Berita sdh basi,0


In [19]:
# 1. Siapkan Stopwords dan Stemmer
stop_words_id = set(stopwords.words('indonesian')) # Jadikan 'set' agar pencarian lebih cepat O(1)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# 2. Fungsi Gabungan Pembersihan Teks
def clean_text(text):
    text = str(text).lower() # Casefolding
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE) # Hapus URL
    text = text.replace('\\t', ' ').replace('\\n', ' ').replace('\\u', ' ').replace('\\', '') # Hapus karakter aneh
    text = re.sub(r"(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])", " ", text) # Hapus tanda baca/mention
    text = re.sub(r"\d+", "", text) # Hapus angka
    text = re.sub(r'\s+', ' ', text).strip() # Rapikan spasi berlebih
    return text

print("Mulai membersihkan teks...")
df['clean_text'] = df['Komen'].apply(clean_text)

# 3. Tokenisasi & Stopword Removal (Digabung agar ringkas)
print("Mulai tokenisasi dan hapus stopwords...")
df['tokens'] = df['clean_text'].apply(lambda x: [word for word in nltk.word_tokenize(x) if word not in stop_words_id])

# 4. Stemming dengan teknik Dictionary (Sangat efisien!)
print("Mulai stemming...")
# Kumpulkan semua kata unik
term_dict = {}
for document in df['tokens']:
    for term in document:
        if term not in term_dict:
            term_dict[term] = stemmer.stem(term) # Stemming hanya dilakukan 1x per kata unik

# Terapkan hasil stemming ke dataframe
df['stemmed_tokens'] = df['tokens'].apply(lambda x: [term_dict[term] for term in x])

# 5. Gabungkan kembali token menjadi kalimat
df['final_text'] = df['stemmed_tokens'].apply(lambda x: ' '.join(x))

df[['Komen', 'final_text']].head()

Mulai membersihkan teks...
Mulai tokenisasi dan hapus stopwords...
Mulai stemming...


,Komen,final_text
0,Saya pastikan kepala kebo akan nyungsep lagi.,pasti kepala kebo nyungsep
1,Singkirkan capres ABG WNI bangsa lain.....pili...,singkir capres abg wni bangsa pilih bangsa asl...
2,PK GNJR PK MAGVOD ... Mntab .. 💪💪💪🔥🔥❤️❤️🤟🤟🤟🤟🤟...,pk gnjr pk magvod mntab
3,29 tetap pilih figurnya. Yang hobbi korupsi pa...,pilih figur hobbi korupsi cm lirik hobbi ngged...
4,Berita sdh basi,berita sdh basi


In [20]:
# 1. TF-IDF
# Hapus stop_words='english' karena teks berbahasa Indonesia dan sudah dibersihkan
tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1,1))
train_data = tfidf.fit_transform(df['final_text'])

# Tampilkan DataFrame TF-IDF (Opsional, awas memori penuh jika datanya puluhan ribu)
# df_tfidf = pd.DataFrame(train_data.toarray(), columns=tfidf.get_feature_names_out())
# print(df_tfidf.head())

# 2. Topic Modeling dengan Truncated SVD (LSA)
num_components = 10
lsa = TruncatedSVD(n_components=num_components, n_iter=100, random_state=42)
lsa.fit_transform(train_data)

# 3. Tampilkan Hasil Topik
terms = tfidf.get_feature_names_out()

print("=== HASIL TOPIC MODELING ===")
for index, component in enumerate(lsa.components_):
    zipped = zip(terms, component)
    # Ambil 5 term dengan bobot tertinggi untuk tiap topik
    top_terms_key = sorted(zipped, key=lambda t: t[1], reverse=True)[:5]
    top_terms_list = [term for term, weight in top_terms_key]

    print(f"Topik {index}: {', '.join(top_terms_list)}")

=== HASIL TOPIC MODELING ===
Topik 0: prabowo, pilih, ganjar, yg, br
Topik 1: prabowo, subianto, all, in, setia
Topik 2: ganjar, pranowo, prabowo, dukung, mahfud
Topik 3: anies, baswedan, ganjar, dukung, ri
Topik 4: anis, baswedan, ganjar, ri, presiden
Topik 5: br, presiden, capres, dukung, mahfud
Topik 6: pilih, br, bagus, capres, banteng
Topik 7: yg, capres, presiden, menang, nya
Topik 8: presiden, pilih, ri, baswedan, pranowo
Topik 9: capres, bagus, presiden, mahfud, nya
